In [1]:
import os
import pandas as pd
import numpy as np
import re

# Thiết lập đường dẫn thư mục linh hoạt
current_dir = os.getcwd()
while not os.path.exists(os.path.join(current_dir, "dataset")) and os.path.dirname(current_dir) != current_dir:
    current_dir = os.path.dirname(current_dir)
BASE_DIR = current_dir

RAW_CSV_DIR = os.path.join(BASE_DIR, "dataset", "raw", "csv")
PROCESSED_DIR = os.path.join(BASE_DIR, "dataset", "processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)

print(f"Thư mục dữ liệu thô: {RAW_CSV_DIR}")
print(f"Thư mục xuất file gộp: {PROCESSED_DIR}")

Thư mục dữ liệu thô: d:\HK5-2026-2027\IS217.R11\dataset\raw\csv
Thư mục xuất file gộp: d:\HK5-2026-2027\IS217.R11\dataset\processed


In [2]:
# Hàm chuẩn hóa tên cột
def normalize_columns_names(col: str) -> str:
    col = col.strip().lower()
    col = re.sub(r'[\s\-]+', '_', col)
    col = re.sub(r'[^\w_]', '', col)
    if col == 'id':
        return 'team_id'
    special_columns = {
        'arenacapacity': 'arena_capacity',
        'generalmanager': 'general_manager',
        'headcoach': 'head_coach',
        'dleagueaffiliation': 'd_league_affiliation',
        'yearfounded': 'year_founded'
    }
    return special_columns.get(col, col)

In [3]:
# 1. Gộp dữ liệu Đội bóng (teams_merged.csv)
# Sử dụng: team.csv, team_details.csv
df_team = pd.read_csv(os.path.join(RAW_CSV_DIR, "team.csv"))
df_team_details = pd.read_csv(os.path.join(RAW_CSV_DIR, "team_details.csv"))

df_team.columns = [normalize_columns_names(col) for col in df_team.columns]
df_team_details.columns = [normalize_columns_names(col) for col in df_team_details.columns]  

common_columns = set(df_team.columns) & set(df_team_details.columns)
df_team_details = df_team_details.drop(columns=list(common_columns - {'team_id'}))
df_team_merged = pd.merge(df_team, df_team_details, on='team_id', how='left')

# Loại bỏ cột mạng xã hội và thông tin thừa không phục vụ OLAP
drop_teams = ['facebook', 'instagram', 'twitter', 'd_league_affiliation', 'nickname']
df_team_merged = df_team_merged.drop(columns=[c for c in drop_teams if c in df_team_merged.columns])

out_teams = os.path.join(PROCESSED_DIR, "teams_merged.csv")
df_team_merged.to_csv(out_teams, index=False)
print(f"Đã gộp teams thành công: {len(df_team_merged)} dòng, {len(df_team_merged.columns)} cột -> {out_teams}")

Đã gộp teams thành công: 30 dòng, 11 cột -> d:\HK5-2026-2027\IS217.R11\dataset\processed\teams_merged.csv


In [4]:
# 2. Gộp dữ liệu Cầu thủ (players_merged.csv)
# Sử dụng: player.csv, common_player_info.csv, draft_history.csv, draft_combine_stats.csv
df_player = pd.read_csv(os.path.join(RAW_CSV_DIR, "player.csv"))
df_info = pd.read_csv(os.path.join(RAW_CSV_DIR, "common_player_info.csv"))
df_draft = pd.read_csv(os.path.join(RAW_CSV_DIR, "draft_history.csv"))
df_combine = pd.read_csv(os.path.join(RAW_CSV_DIR, "draft_combine_stats.csv"))

df_player = df_player.rename(columns={'id': 'player_id'})
df_info = df_info.rename(columns={'person_id': 'player_id'})
df_draft = df_draft.rename(columns={'person_id': 'player_id'})

# common_player_info: loại bỏ cột trùng tên với player.csv
info_cols_to_drop = ['first_name', 'last_name', 'display_first_last', 'display_last_comma_first', 'display_fi_last', 'player_slug']
df_info_clean = df_info.drop(columns=[c for c in info_cols_to_drop if c in df_info.columns])
df_info_clean = df_info_clean.drop_duplicates(subset=['player_id'])

# draft_history: đổi tên cột rõ nghĩa
df_draft_clean = df_draft[['player_id', 'season', 'round_number', 'round_pick', 'overall_pick', 'organization', 'organization_type']].copy()
df_draft_clean = df_draft_clean.rename(columns={
    'season': 'draft_season',
    'round_number': 'draft_round_num',
    'round_pick': 'draft_round_pick',
    'overall_pick': 'draft_overall_pick',
    'organization': 'draft_college_or_org',
    'organization_type': 'draft_org_type'
})
df_draft_clean = df_draft_clean.drop_duplicates(subset=['player_id'])

# draft_combine_stats: chỉ số đo lường thể chất
combine_cols = ['player_id', 'height_wo_shoes', 'height_w_shoes', 'wingspan', 'standing_reach', 'body_fat_pct', 'standing_vertical_leap', 'max_vertical_leap', 'lane_agility_time', 'three_quarter_sprint', 'bench_press']
df_combine_clean = df_combine[[c for c in combine_cols if c in df_combine.columns]].copy()
df_combine_clean = df_combine_clean.drop_duplicates(subset=['player_id'])

# Gộp tuần tự vào bảng gốc player
df_players_merged = df_player.copy()
df_players_merged = pd.merge(df_players_merged, df_info_clean, on='player_id', how='left')
df_players_merged = pd.merge(df_players_merged, df_draft_clean, on='player_id', how='left')
df_players_merged = pd.merge(df_players_merged, df_combine_clean, on='player_id', how='left')

# Loại bỏ cột thừa không phục vụ Dimension Player
drop_players = [
    'first_name', 'last_name', 'playercode', 'dleague_flag', 'nba_flag', 'games_played_flag', 
    'games_played_current_season_flag', 'greatest_75_flag', 'team_id', 'team_name', 'team_abbreviation', 
    'team_code', 'team_city', 'draft_year', 'draft_round', 'draft_number', 'last_affiliation', 
    'rosterstatus', 'height_wo_shoes', 'standing_reach', 'body_fat_pct', 'lane_agility_time', 
    'three_quarter_sprint', 'standing_vertical_leap'
]
df_players_merged = df_players_merged.drop(columns=[c for c in drop_players if c in df_players_merged.columns])

# Chuẩn hóa dữ liệu theo chuẩn Pipeline cho SSIS:
# 1. Khử dấu phẩy trong tên trường học để tránh làm lệch cột CSV khi SSIS đọc
if 'school' in df_players_merged.columns:
    df_players_merged['school'] = df_players_merged['school'].astype(str).str.replace(',', ' -')

# 2. Định dạng ngày sinh chuẩn YYYY-MM-DD
if 'birthdate' in df_players_merged.columns:
    df_players_merged['birthdate'] = pd.to_datetime(df_players_merged['birthdate'], errors='coerce').dt.strftime('%Y-%m-%d')

# 3. Ép kiểu số nguyên cho các cột có chứa giá trị trống (tránh bị dính đuôi .0)
int_cols = ['from_year', 'to_year', 'draft_round_num', 'draft_overall_pick']
for col in int_cols:
    if col in df_players_merged.columns:
        df_players_merged[col] = df_players_merged[col].astype('Int64')

out_players = os.path.join(PROCESSED_DIR, "players_merged.csv")
df_players_merged.to_csv(out_players, index=False)
print(f"Đã gộp players thành công: {len(df_players_merged)} dòng, {len(df_players_merged.columns)} cột -> {out_players}")

Đã gộp players thành công: 4831 dòng, 23 cột -> d:\HK5-2026-2027\IS217.R11\dataset\processed\players_merged.csv


In [5]:
# 3. Gộp dữ liệu Trận đấu (games_merged.csv)
# Sử dụng: game.csv, game_info.csv, game_summary.csv, other_stats.csv, line_score.csv
df_game = pd.read_csv(os.path.join(RAW_CSV_DIR, "game.csv"))
df_game = df_game.drop_duplicates(subset=['game_id']).copy()

df_info = pd.read_csv(os.path.join(RAW_CSV_DIR, "game_info.csv"))
df_summary = pd.read_csv(os.path.join(RAW_CSV_DIR, "game_summary.csv"))
df_other = pd.read_csv(os.path.join(RAW_CSV_DIR, "other_stats.csv"))
df_line_score = pd.read_csv(os.path.join(RAW_CSV_DIR, "line_score.csv"))

df_info = df_info[['game_id', 'attendance', 'game_time']].drop_duplicates(subset=['game_id'])
df_summary = df_summary[['game_id', 'game_status_text', 'natl_tv_broadcaster_abbreviation']].copy()
df_summary = df_summary.rename(columns={'natl_tv_broadcaster_abbreviation': 'tv_broadcaster'}).drop_duplicates(subset=['game_id'])

other_cols = [
    'game_id', 'pts_paint_home', 'pts_2nd_chance_home', 'pts_fb_home', 'pts_off_to_home', 'largest_lead_home',
    'pts_paint_away', 'pts_2nd_chance_away', 'pts_fb_away', 'pts_off_to_away', 'largest_lead_away',
    'lead_changes', 'times_tied'
]
df_other = df_other[other_cols].drop_duplicates(subset=['game_id'])

line_cols = [
    'game_id', 'pts_qtr1_home', 'pts_qtr2_home', 'pts_qtr3_home', 'pts_qtr4_home', 'pts_ot1_home',
    'pts_qtr1_away', 'pts_qtr2_away', 'pts_qtr3_away', 'pts_qtr4_away', 'pts_ot1_away'
]
df_line_score = df_line_score[line_cols].drop_duplicates(subset=['game_id'])

# Gộp tuần tự vào df_game
df_games_merged = df_game.copy()
df_games_merged = pd.merge(df_games_merged, df_info, on='game_id', how='left')
df_games_merged = pd.merge(df_games_merged, df_summary, on='game_id', how='left')
df_games_merged = pd.merge(df_games_merged, df_other, on='game_id', how='left')
df_games_merged = pd.merge(df_games_merged, df_line_score, on='game_id', how='left')

# Giữ các cột chuẩn cho Fact Team Game (loại metadata và tỷ lệ % semi-additive theo chuẩn Kimball)
keep_games = [
    'game_id', 'game_date', 'season_id', 'season_type',
    'team_id_home', 'team_id_away',
    'wl_home', 'pts_home', 'plus_minus_home', 
    'fgm_home', 'fga_home', 'fg3m_home', 'fg3a_home', 
    'reb_home', 'ast_home', 'tov_home', 'pts_paint_home', 'pts_fb_home',
    'wl_away', 'pts_away', 'plus_minus_away', 
    'fgm_away', 'fga_away', 'fg3m_away', 'fg3a_away', 
    'reb_away', 'ast_away', 'tov_away', 'pts_paint_away', 'pts_fb_away'
]
df_games_merged = df_games_merged[keep_games]

# Lọc bảo đảm toàn vẹn khóa ngoại (FK): Chỉ giữ các trận đấu thuộc 30 đội bóng chính thức
valid_team_ids = set(pd.read_csv(os.path.join(PROCESSED_DIR, "teams_merged.csv"))['team_id'])
before_games = len(df_games_merged)
df_games_merged = df_games_merged[
    df_games_merged['team_id_home'].isin(valid_team_ids) & 
    df_games_merged['team_id_away'].isin(valid_team_ids)
]
after_games = len(df_games_merged)

out_games = os.path.join(PROCESSED_DIR, "games_merged.csv")
df_games_merged.to_csv(out_games, index=False)
print(f"Đã gộp games thành công: {after_games:,} dòng (đã loại {before_games - after_games:,} trận của 52 đội đã giải thể), {len(df_games_merged.columns)} cột -> {out_games}")

Đã gộp games thành công: 65642 dòng, 30 cột -> d:\HK5-2026-2027\IS217.R11\dataset\processed\games_merged.csv


In [6]:
# 4. Trích xuất liên kết Cầu thủ ↔ Trận đấu (player_game.csv) từ play_by_play.csv (2.2GB)
# Xử lý theo chunk, loại bỏ data bẩn và kiểm tra toàn vẹn khóa ngoại (FK) với dim_player, dim_team, games_merged
CHUNK_SIZE = 1_000_000
player_cols = [
    ('player1_id', 'player1_team_id'),
    ('player2_id', 'player2_team_id'),
    ('player3_id', 'player3_team_id'),
]

frames = []
for chunk in pd.read_csv(
    os.path.join(RAW_CSV_DIR, "play_by_play.csv"),
    usecols=['game_id'] + [c for pair in player_cols for c in pair],
    chunksize=CHUNK_SIZE
):
    parts = []
    for pcol, tcol in player_cols:
        mask = chunk[pcol].notna() & (chunk[pcol] != 0)
        sub = chunk.loc[mask, ['game_id', pcol, tcol]].copy()
        sub.columns = ['game_id', 'player_id', 'team_id']
        parts.append(sub)
    combined = pd.concat(parts).drop_duplicates()
    frames.append(combined)

df_player_game = pd.concat(frames).drop_duplicates()

# Danh sách khóa chính hợp lệ từ dim_player, dim_team và games_merged đã làm sạch
valid_player_ids = set(pd.read_csv(os.path.join(PROCESSED_DIR, "players_merged.csv"))['player_id'])
valid_team_ids = set(pd.read_csv(os.path.join(PROCESSED_DIR, "teams_merged.csv"))['team_id'])
valid_game_ids = set(df_games_merged['game_id'].astype(str))

before = len(df_player_game)

# Lọc bỏ data bẩn:
# 1. Bỏ player_id thực chất là team event (> 1.6 tỷ) và team_id null
df_player_game = df_player_game[df_player_game['player_id'] <= 1_610_000_000]
df_player_game = df_player_game[df_player_game['team_id'].notna()]

# 2. Ép kiểu dữ liệu sang int
df_player_game['player_id'] = df_player_game['player_id'].astype(int)
df_player_game['team_id'] = df_player_game['team_id'].astype(int)

# 3. Đảm bảo toàn vẹn khóa ngoại (FK) với dim_player, dim_team và fact_team_game
df_player_game = df_player_game[df_player_game['player_id'].isin(valid_player_ids)]
df_player_game = df_player_game[df_player_game['team_id'].isin(valid_team_ids)]
df_player_game = df_player_game[df_player_game['game_id'].astype(str).isin(valid_game_ids)]

after = len(df_player_game)

out_player_game = os.path.join(PROCESSED_DIR, "player_game.csv")
df_player_game.to_csv(out_player_game, index=False)

print(f"Trích xuất player_game thành công: {after:,} dòng (đã loại {before - after:,} dòng bẩn & không khớp FK)")
print(f"  - Unique game_id : {df_player_game['game_id'].nunique():,}")
print(f"  - Unique player_id: {df_player_game['player_id'].nunique():,}")
print(f"  - Avg players/game: {df_player_game.groupby('game_id')['player_id'].nunique().mean():.1f}")
print(f"  → {out_player_game}")

Trích xuất player_game thành công: 617,110 dòng (đã loại 87,759 dòng bẩn)
  - Unique game_id : 29,798
  - Unique player_id: 2,559
  - Avg players/game: 20.7
  → d:\HK5-2026-2027\IS217.R11\dataset\processed\player_game.csv
